In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql://tongli1997@db.pcic.uvic.ca:5433/crmp?keepalives=1&keepalives_idle=300&keepalives_interval=300&keepalives_count=9&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

### Fix problem

In [4]:
path = '/workspaces/crmprtd/update_sheet/sheet_24Dec/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges (2).xlsx', header = 1)   # pandas automatically uses openpyxl
df =  df[['ISSUE','Station ID', 'Network ID', 'Network Name', 'Native ID', 'Unique Names', 'Unique Location (String)']].reset_index(drop=True)
df["Network ID"] = pd.to_numeric(df["Network ID"], errors="coerce").astype("Int64")
df["Station ID"] = pd.to_numeric(df["Station ID"], errors="coerce").astype("Int64")


df_miss = df[
    (df["ISSUE"].str.strip().str.lower() == "missing data") 
]

df_miss

wanted_issues = [
"NOT on ENV LIST",
"PARKS CANADA"]

df_list = df[
    df["ISSUE"].str.strip().isin(wanted_issues) 
]

df_combined = pd.concat([df_miss, df_list], ignore_index=True)

df_combined



,ISSUE,Station ID,Network ID,Network Name,Native ID,Unique Names,Unique Location (String)
0,Missing data,12626,36,CRD-WS,HY002,Judge Creek,"123.673611 W, 48.585278 N, Elev. 200 m"
1,Missing data,12625,36,CRD-WS,FW002,Rithet Creek,"123.714167 W, 48.5675 N, Elev. 223 m"
2,Missing Data,12203,9,BC-Air -> MVRD-AIR,E308566 -> T046,New Westminster Sapperton Park -> Sapperton Park,"122.894487 W, 49.227045 N, Elev. 45 m -> 122.8..."
3,Missing Data,12205,9,BC-Air -> MVRD-AIR,E207723 -> T013,North Delta,"122.9017 W, 49.1583 N, Elev. 111 m"
4,NOT on ENV LIST,2486,5,BCH-GSO-HMP,OMI,Ominica R. ab Osilinka R. -> Omineca River abo...,"124.56 W, 55.94 N, Elev. 715 m"
5,NOT on ENV LIST,2488,5,BCH-GSO-HMP,OSP,Ospika R. ab Alley Ck -> Ospika River above Al...,"123.93 W, 56.46 N, Elev. 750 m"
6,PARKS CANADA,2458,5,BCH-GSO-HMP,FDL,Fidelty Mtn. -> Fidelity Mountain,"117.7006167 W, 51.23741667 N, Elev. 1800 m"


### update native_id

In [5]:
df_native_id_up = df_combined[df_combined['Native ID'].str.contains("->", na=False)][['Station ID', 'Network ID', 'Native ID',]]



split_ids = df_native_id_up['Native ID'].astype(str).str.split('->', expand=True)

df_native_id_up['old_native_id'] = split_ids[0].str.strip()
df_native_id_up['new_native_id'] = split_ids[1].str.strip()


df_native_id_up

,Station ID,Network ID,Native ID,old_native_id,new_native_id
2,12203,9,E308566 -> T046,E308566,T046
3,12205,9,E207723 -> T013,E207723,T013


In [6]:
select_sql = sa.text("""
SELECT DISTINCT
    n.network_id
FROM meta_station s
JOIN meta_network n
  ON n.network_id = s.network_id
JOIN meta_history m
  ON m.station_id = s.station_id
WHERE s.native_id = :native_id
""")

with engine.begin() as conn:
    for idx, row in df_native_id_up.iterrows():

        native_id = row["new_native_id"]

        result = conn.execute(
            select_sql,
            {"native_id": native_id}
        ).fetchall()

        if not result:
            print(f"❌ native_id {native_id} not found")
            continue

        df_native_id_up.at[idx, "new_network_id"] = ",".join(
              map(str, {r.network_id for r in result})
        )

df_native_id_up

❌ native_id T046 not found
❌ native_id T013 not found


,Station ID,Network ID,Native ID,old_native_id,new_native_id
2,12203,9,E308566 -> T046,E308566,T046
3,12205,9,E207723 -> T013,E207723,T013


### update station name

In [7]:
df_staion_name_up = df_combined[df_combined['Unique Names'].str.contains("->", na=False)][['Station ID', 'Network ID', 'Unique Names',]]



split_ids = df_staion_name_up['Unique Names'].astype(str).str.split('->', expand=True)

df_staion_name_up['old_name'] = split_ids[0].str.strip()
df_staion_name_up['new_name'] = split_ids[1].str.strip()


df_staion_name_up

,Station ID,Network ID,Unique Names,old_name,new_name
2,12203,9,New Westminster Sapperton Park -> Sapperton Park,New Westminster Sapperton Park,Sapperton Park
4,2486,5,Ominica R. ab Osilinka R. -> Omineca River abo...,Ominica R. ab Osilinka R.,Omineca River above Osilinka River
5,2488,5,Ospika R. ab Alley Ck -> Ospika River above Al...,Ospika R. ab Alley Ck,Ospika River above Alley Creek
6,2458,5,Fidelty Mtn. -> Fidelity Mountain,Fidelty Mtn.,Fidelity Mountain


In [8]:
check_sql = sa.text("""
SELECT m.station_id, m.station_name
FROM meta_history m
JOIN meta_station s ON m.station_id = s.station_id
WHERE s.station_id = :station_id
""")

import numpy as np

with engine.begin() as conn:
    for _, row in df_staion_name_up.iterrows():

        station_id = int(row["Station ID"])

        old_name  = row["old_name"]
        new_name  = row["new_name"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 2. Compare with expected OLD values ---

        if db_row.station_name != old_name:
            print(
                f"⚠️ Station {station_id} (DB, XLS values differ)\n"
                f"   DB : station name is {db_row.station_name}\n"
                f"   XLS: station name is {old_name}"
            )
        else:
            print(
                f"✅ Station {station_id}, {db_row.station_name}, the names match "
            )


✅ Station 12203, New Westminster Sapperton Park, the names match 
✅ Station 2486, Ominica R. ab Osilinka R., the names match 
✅ Station 2488, Ospika R. ab Alley Ck, the names match 
✅ Station 2458, Fidelty Mtn., the names match 


### update location

In [9]:
df_loc_up = df_combined[df_combined['Unique Location (String)'].str.contains("->", na=False)][['Station ID', 'Network ID', 'Unique Location (String)',]]

df_loc_up

import re


def _parse_single_location(loc_str):
    """
    Parse:
    '123.764438 W, 48.51616 N, Elev. 512.14 m'
    '123.764438 W, 48.51616 N, Elev. null m'

    Returns: (lat, lon, elev)
    """
    if pd.isna(loc_str):
        return np.nan, np.nan, np.nan

    pattern = (
        r"([\d\.]+)\s*([EW]),\s*"
        r"([\d\.]+)\s*([NS]),\s*"
        r"Elev\.\s*(null|[\d\.]+)\s*m"
    )

    m = re.search(pattern, loc_str, re.IGNORECASE)
    if not m:
        return np.nan, np.nan, np.nan

    lon_val, lon_dir, lat_val, lat_dir, elev_val = m.groups()

    lon = float(lon_val) * (-1 if lon_dir.upper() == "W" else 1)
    lat = float(lat_val) * (-1 if lat_dir.upper() == "S" else 1)

    elev = np.nan if elev_val.lower() == "null" else float(elev_val)

    return lat, lon, elev


def parse_old_new_lat_lon_elev(loc_str):
    """
    Returns:
    (old_lat, old_lon, old_elev, new_lat, new_lon, new_elev)
    """
    if pd.isna(loc_str):
        return (np.nan,) * 6

    if "->" in loc_str:
        old_str, new_str = map(str.strip, loc_str.split("->", 1))
    else:
        old_str = new_str = loc_str.strip()

    old_lat, old_lon, old_elev = _parse_single_location(old_str)
    new_lat, new_lon, new_elev = _parse_single_location(new_str)

    return (
        old_lat, old_lon, old_elev,
        new_lat, new_lon, new_elev
    )

cols = [
    'old_lat', 'old_lon', 'old_elev',
    'new_lat', 'new_lon', 'new_elev'
]

df_loc_up[cols] = df_loc_up['Unique Location (String)'].apply(
    lambda x: pd.Series(parse_old_new_lat_lon_elev(x))
)

# 
df_loc_up = df_loc_up.drop(columns=['Unique Location (String)'])

df_loc_up

,Station ID,Network ID,old_lat,old_lon,old_elev,new_lat,new_lon,new_elev
2,12203,9,49.227045,-122.894487,45.0,49.227045,-122.894449,45.0


In [10]:
import numpy as np

update_sql = sa.text("""
UPDATE meta_history
SET
    lat  = :new_lat,
    lon  = :new_lon,
    elev = :new_elev
WHERE station_id = :station_id
""")

with engine.begin() as conn:
    for _, row in df_loc_up.iterrows():

        station_id = int(row["Station ID"])

        old_lat  = row["old_lat"]
        old_lon  = row["old_lon"]
        old_elev = row["old_elev"]

        new_lat  = row["new_lat"]
        new_lon  = row["new_lon"]
        new_elev = row["new_elev"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 3. Perform update ---
        result = conn.execute(
            update_sql,
            {
                "station_id": station_id,
                "new_lat": new_lat,
                "new_lon": new_lon,
                "new_elev": new_elev,
            }
        )

        if result.rowcount == 1:
            print(
                f"✅ Updated station {station_id}: "
                f"({old_lat}, {old_lon}, {old_elev}) → "
                f"({new_lat}, {new_lon}, {new_elev})"
            )
        else:
            print(f"⚠️ Station {station_id}: no update applied")

✅ Updated station 12203: (49.227045, -122.894487, 45.0) → (49.227045, -122.8944487, 45.0)
